# M2: Audit Leakage untuk Varian Eksperimen

Kelompok 03 · Titut Encoder · IF25-40304 Pembelajaran Mesin Multimodal · Peran 1 (Data dan Etika Data)

Notebook ini memeriksa apakah angka tiap varian M2 jujur, atau menggelembung karena kebocoran data. Isinya:

1. **Audit data** (hasil `src/audit_leakage.py`): caption persis sama, caption mirip, dan gambar near-duplicate antara train dan val.
2. **Audit rancangan** per varian: apa yang dilatih, apa yang dipilih dengan melihat val.
3. **Sensitivitas**: semua varian pada val penuh (275) dan val bersih, dengan bootstrap CI.
4. **Ablation filter**: proyeksi dilatih tanpa filter, dengan filter caption saja, dan dengan filter audit.

**Cara menjalankan di Colab**
- **Run pertama:** pilih *Runtime → Change runtime type → GPU (T4)*. Embedding yang belum ada di Drive dihitung sekali (encoder tetap beku) lalu disimpan ke `MyDrive/Dataset-Tubes-Multimodal/results/m2/`. Di CPU langkah ini sekitar 75 menit; di GPU jauh lebih cepat.
- **Run berikutnya:** CPU cukup. Embedding dimuat dari Drive, yang dilatih hanya proyeksi kecil (detik per seed).
- Kode yang sama berjalan di CPU (aturan TOR); GPU hanya mempercepat.

*Data: SEA-VL Indonesian subset, CC BY-SA 4.0 (Cahyawijaya et al., 2025).*

## 0. Persiapan

In [ ]:
import os, sys, re, json, random, time, subprocess, zipfile
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    display = print

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    if os.path.exists("/content/Tubes-Multimodal"):
        os.chdir("/content/Tubes-Multimodal")
        subprocess.run(["git", "pull"], check=False)
    else:
        os.chdir("/content")
        subprocess.run(["git", "clone", "https://github.com/bayybrigas04/Tubes-Multimodal.git"], check=True)
        os.chdir("/content/Tubes-Multimodal")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/Dataset-Tubes-Multimodal"   # tempat cache embedding disimpan
    DATA_DIR = "/content/data/bundle-mahasiswa"
    if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
        import glob
        kandidat = [f"{DRIVE_DIR}/bundle-mahasiswa.zip"]
        for pola in ("/content/drive/MyDrive/bundle-mahasiswa*.zip", "/content/drive/MyDrive/*/bundle-mahasiswa*.zip",
                     "/content/drive/MyDrive/*/*/bundle-mahasiswa*.zip", "/content/bundle-mahasiswa*.zip"):
            kandidat += sorted(glob.glob(pola))
        ZIP_PATH = next((p for p in kandidat if os.path.isfile(p)), None)
        if ZIP_PATH is None:
            raise FileNotFoundError(
                "bundle-mahasiswa.zip tidak ditemukan di Google Drive.\n"
                "Unggah bundle-mahasiswa.zip (dari LMS) ke 'MyDrive/Dataset-Tubes-Multimodal/' lalu jalankan sel ini lagi.")
        print("Dataset zip:", ZIP_PATH)
        os.makedirs("/content/data", exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall("/content/data")
    assert os.path.exists(os.path.join(DATA_DIR, "train.csv")), f"train.csv tidak ada di {DATA_DIR}"
    CACHE_DIR = f"{DRIVE_DIR}/results/m2"          # cache embedding dari peran 3 (ubah jika berbeda)
    M1_VAL    = f"{DRIVE_DIR}/results/m1/cache_val"
else:
    DATA_DIR  = os.environ.get("DATA_DIR", "data")
    CACHE_DIR = os.environ.get("CACHE_DIR", "results/m2")
    M1_VAL    = os.environ.get("M1_VAL", "results/m1/cache_val")

sys.path.insert(0, ".")
AUDIT_DIR = Path("results/m2_audit")
SEED, SEEDS_PROJ, KS = 42, [42, 123, 456], (1, 5, 10)
ARAH = ["gambar->teks-ID", "gambar->teks-EN", "teks-ID->gambar", "teks-EN->gambar"]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", IN_COLAB, "| device:", device)
print("DATA_DIR :", DATA_DIR)
print("CACHE_DIR:", CACHE_DIR)

## 1. Data dan filter dari audit tahap 1

Tiga kondisi filter train yang dibandingkan:

| Kondisi | Aturan |
|---|---|
| tanpa filter | semua 2.072 baris train |
| filter caption (peran 3) | buang baris train yang caption EN atau ID-nya sama dengan val |
| filter audit (peran 1) | filter caption + gambar near-duplicate (hash perseptual dan cosine CLIP) |

In [ ]:
def norm_caption(s):
    return re.sub(r"[^\w\s]", "", str(s).lower()).strip()

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
N_train, N_val = len(train), len(val)
filt = pd.read_csv(AUDIT_DIR / "leakage_filter.csv")

# kondisi 1: tanpa filter
keep_none = np.ones(N_train, dtype=bool)
# kondisi 2: filter caption saja (aturan yang dipakai peran 3)
_vc = set(norm_caption(c) for c in val["caption_id"]) | set(norm_caption(c) for c in val["caption_en"])
keep_caption = np.array([(norm_caption(i) not in _vc) and (norm_caption(e) not in _vc)
                         for i, e in zip(train["caption_id"], train["caption_en"])])
# kondisi 3: filter audit (caption + gambar near-duplicate)
keep_audit = ~train["image_id"].isin(set(filt.loc[filt["keluarkan_dari_train"], "image_id"])).to_numpy()
# val bersih: buang baris val yang berstatus pasti bocor
val_bersih = ~val["image_id"].isin(set(filt.loc[filt["val_bocor"], "image_id"])).to_numpy()

KONDISI = {"tanpa filter": keep_none, "filter caption (peran 3)": keep_caption, "filter audit (peran 1)": keep_audit}
train_group = train["caption_id"].map(norm_caption).values

print(f"Train {N_train} | Val {N_val}")
for k, m in KONDISI.items():
    print(f"  {k:28s}: train dipakai {int(m.sum())} (dibuang {int((~m).sum())})")
print(f"  val bersih: {int(val_bersih.sum())} dari {N_val} (13 baris val berstatus pasti bocor)")
print("Selisih filter audit vs caption:",
      sorted(set(train.image_id[~keep_audit]) - set(train.image_id[~keep_caption])))

In [ ]:
# Ringkasan temuan audit tahap 1 (dari results/m2_audit/)
print(filt.groupby(["split", "status"]).size().unstack(fill_value=0))

pas = pd.read_csv(AUDIT_DIR / "pasangan_mirip.csv")
print("\nPasangan val-train berstatus 'pasti' yang BUKAN caption persis sama (gambar near-duplicate):")
display(pas[(pas["tingkat"] == "pasti") & (~pas["persis"])][["val_image", "train_image", "alasan", "hamming_min", "cos_gambar"]])

# Bukti visual: foto yang sama dengan resolusi berbeda (lolos dari pemeriksaan MD5 di M1)
a = val[val.image_id == "svl_143496781"].iloc[0]
b = train[train.image_id == "svl_141004060"].iloc[0]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for x, r, t in zip(ax, (a, b), ("val svl_143496781", "train svl_141004060")):
    x.imshow(Image.open(os.path.join(DATA_DIR, r["file"])).convert("RGB")); x.set_title(t); x.axis("off")
plt.suptitle("Gambar duplikat lintas split (resolusi berbeda, MD5 berbeda, caption berbeda)")
plt.tight_layout(); plt.show()

## 2. Audit rancangan per varian

| Varian | Yang dilatih / dipilih | Data yang disentuh | Risiko leakage | Pemeriksaan | Temuan |
|---|---|---|---|---|---|
| Baseline M1 | tidak ada | val (evaluasi saja) | rendah | baseline direproduksi, cocok dengan M1 | bersih |
| SigLIP 2 Base | tidak ada, encoder beku | val (evaluasi saja) | **kontaminasi pra-latih** (WebLI) | tidak dapat diperiksa dari sini | limitasi |
| NLLB-CLIP-SigLIP | tidak ada, encoder beku | val (evaluasi saja) | kontaminasi pra-latih | tidak dapat diperiksa | limitasi |
| Baseline + proyeksi | proyeksi residual; early stopping dari dev (15% train) | train, tanpa val | baris train overlap val | ablation filter (bagian 5) | lihat hasil |
| Encoder terbaik + proyeksi | seperti di atas; **encoder dipilih dengan R@1 val** | train, val (pemilihan encoder) | tuning di val | selisih SigLIP 2 vs NLLB jauh di atas CI | dilaporkan peran 3 |

**Perlu dikonfirmasi peran 3:** hyperparameter proyeksi (lr, suhu, patience, batch, weight decay) ditetapkan tanpa melihat skor val.

## 3. Siapkan embedding (muat dari Drive, atau hitung bila belum ada)

Model dan cara encode sama dengan `m2_perbandingan.ipynb` (peran 3): `clip-ViT-B-32` + `multilingual-v1`, `ViT-B-16-SigLIP2-256` (webli), dan `nllb-clip-base-siglip` dengan `set_language`. Encoder dibekukan; tidak ada pelatihan di sel ini.

In [ ]:
import importlib.util

SUMBER = {   # nama: (cache val, cache train)
    "Baseline M1":      (M1_VAL,                                       os.path.join(CACHE_DIR, "cache_baseline_train")),
    "SigLIP 2 Base":    (os.path.join(CACHE_DIR, "cache_siglip2_val"), os.path.join(CACHE_DIR, "cache_siglip_2_base_train")),
    "NLLB-CLIP-SigLIP": (os.path.join(CACHE_DIR, "cache_nllb_val"),    None),   # NLLB tidak dilatih proyeksinya
}
LEWATI = set(filter(None, os.environ.get("LEWATI_ENCODER", "").split(",")))   # kosong di Colab

def ada(path, kunci):
    return path is not None and all(os.path.exists(os.path.join(path, f"{k}.npy")) for k in kunci)

def simpan(path, **arr):
    os.makedirs(path, exist_ok=True)
    for k, v in arr.items():
        np.save(os.path.join(path, f"{k}.npy"), v)

def _l2n(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

def batch_gambar(df, bs=32):
    for i in range(0, len(df), bs):
        yield [Image.open(os.path.join(DATA_DIR, f)).convert("RGB") for f in df["file"].iloc[i:i + bs]]

def perlu(modul, paket):
    if importlib.util.find_spec(modul) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paket], check=True)

_M = {}
def hitung_baseline(df, dengan_te):
    perlu("sentence_transformers", "sentence-transformers")
    from sentence_transformers import SentenceTransformer
    if "m1" not in _M:
        _M["m1"] = (SentenceTransformer("sentence-transformers/clip-ViT-B-32", device=device),
                    SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1", device=device))
    m_img, m_txt = _M["m1"]
    out = {"img": np.concatenate([m_img.encode(b, batch_size=32, normalize_embeddings=True) for b in batch_gambar(df)]),
           "ti": m_txt.encode(list(df["caption_id"]), batch_size=64, normalize_embeddings=True)}
    if dengan_te:
        out["te"] = m_img.encode(list(df["caption_en"]), batch_size=64, normalize_embeddings=True)
    return out

def hitung_siglip2(df, dengan_te):
    perlu("open_clip", "open_clip_torch")
    import open_clip
    if "siglip2" not in _M:
        m, _, pre = open_clip.create_model_and_transforms("ViT-B-16-SigLIP2-256", pretrained="webli")
        _M["siglip2"] = (m.eval().to(device), pre, open_clip.get_tokenizer("ViT-B-16-SigLIP2-256"))
    m, pre, tok = _M["siglip2"]
    enc = lambda caps: np.concatenate([m.encode_text(tok(caps[i:i + 64]).to(device)).cpu().numpy()
                                       for i in range(0, len(caps), 64)])
    with torch.no_grad():
        img = np.concatenate([m.encode_image(torch.stack([pre(x) for x in b]).to(device)).cpu().numpy()
                              for b in batch_gambar(df)])
        out = {"img": _l2n(img), "ti": _l2n(enc(list(df["caption_id"])))}
        if dengan_te:
            out["te"] = _l2n(enc(list(df["caption_en"])))
    return out

def hitung_nllb(df):
    perlu("open_clip", "open_clip_torch")
    import open_clip
    if "nllb" not in _M:
        m, pre = open_clip.create_model_from_pretrained("nllb-clip-base-siglip", "v1", device=device)
        _M["nllb"] = (m.eval(), pre, open_clip.get_tokenizer("nllb-clip-base-siglip"))
    m, pre, tok = _M["nllb"]
    def enc(caps, lang):
        tok.set_language(lang)
        return _l2n(np.concatenate([m.encode_text(torch.stack([tok(t) for t in caps[i:i + 64]]).squeeze(1).to(device)).cpu().numpy()
                                    for i in range(0, len(caps), 64)]))
    with torch.no_grad():
        img = np.concatenate([m.encode_image(torch.stack([pre(x) for x in b]).to(device)).cpu().numpy()
                              for b in batch_gambar(df)])
        return {"img": _l2n(img), "ti": enc(list(df["caption_id"]), "ind_Latn"), "te": enc(list(df["caption_en"]), "eng_Latn")}

TUGAS = [  # (encoder, bagian, fungsi)
    ("Baseline M1",      "val",   lambda: hitung_baseline(val, True)),
    ("Baseline M1",      "train", lambda: hitung_baseline(train, False)),
    ("SigLIP 2 Base",    "val",   lambda: hitung_siglip2(val, True)),
    ("SigLIP 2 Base",    "train", lambda: hitung_siglip2(train, False)),
    ("NLLB-CLIP-SigLIP", "val",   lambda: hitung_nllb(val)),
]
WAKTU = {}
for nama, bagian, fn in TUGAS:
    path = SUMBER[nama][0 if bagian == "val" else 1]
    kunci = ("img", "ti", "te") if bagian == "val" else ("img", "ti")
    if ada(path, kunci):
        print(f"[cache ada] {nama} {bagian}: {path}")
        continue
    if nama in LEWATI:
        print(f"[dilewati] {nama} {bagian}")
        continue
    print(f"[hitung]    {nama} {bagian} ...", flush=True)
    t0 = time.time()
    simpan(path, **fn())
    WAKTU[f"{nama} {bagian}"] = round(time.time() - t0, 1)
    print(f"            selesai {WAKTU[f'{nama} {bagian}']}s, disimpan ke {path}")
_M.clear()
print("Waktu hitung embedding (detik):", WAKTU or "semua dari cache")

In [ ]:
def muat_cache(path, kunci):
    if not ada(path, kunci):
        return None
    return {k: np.load(os.path.join(path, f"{k}.npy")) for k in kunci}

VAL_EMB, TRAIN_EMB = {}, {}
for nama, (pv, pt) in SUMBER.items():
    v = muat_cache(pv, ("img", "ti", "te"))
    t = muat_cache(pt, ("img", "ti"))
    if v is None:
        print(f"[LEWATI] {nama}: cache val tidak ditemukan di {pv}")
        continue
    assert len(v["img"]) == N_val, f"{nama}: cache val ({len(v['img'])}) != val.csv ({N_val})"
    VAL_EMB[nama] = v
    if t is not None:
        assert len(t["img"]) == N_train, f"{nama}: cache train ({len(t['img'])}) != train.csv ({N_train})"
        TRAIN_EMB[nama] = t
    print(f"[OK] {nama}: val {v['img'].shape}" + (f", train {t['img'].shape}" if t is not None else ", tanpa cache train"))

## 4. Fungsi evaluasi dan pelatihan proyeksi
Sama dengan notebook peran 3 (`m2_perbandingan.ipynb`) agar hasilnya sebanding; hanya filter train yang divariasikan.

In [ ]:
def ranks_of_truth(A, B):
    sim = A @ B.T
    return (sim > np.diag(sim)[:, None]).sum(1) + 1

def evaluasi(img, ti, te, mask=None):
    if mask is not None:
        img, ti, te = img[mask], ti[mask], te[mask]
    pasangan = {ARAH[0]: (img, ti), ARAH[1]: (img, te), ARAH[2]: (ti, img), ARAH[3]: (te, img)}
    return {a: ranks_of_truth(A, B) for a, (A, B) in pasangan.items()}

def hit1(r): return (r == 1).astype(float)

def ci_mean(x, B=10000, seed=0):
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, len(x), (B, len(x)))].mean(1)
    return float(x.mean()), float(np.quantile(m, 0.025)), float(np.quantile(m, 0.975))

def l2n(x): return x / np.linalg.norm(x, axis=-1, keepdims=True)

class ProyeksiResidual(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.W = nn.Linear(dim, dim, bias=True)
        nn.init.zeros_(self.W.weight); nn.init.zeros_(self.W.bias)
    def forward(self, x):
        y = x + self.W(x)
        return y / y.norm(dim=-1, keepdim=True)

def info_nce(a, b, T=0.07):
    logits = a @ b.T / T
    lab = torch.arange(len(logits), device=logits.device)
    return (nn.functional.cross_entropy(logits, lab) + nn.functional.cross_entropy(logits.T, lab)) / 2

def split_dev(teks, gambar, groups, dev_frac=0.15, seed=42):
    rng = np.random.RandomState(seed)
    uniq = np.array(sorted(set(groups))); rng.shuffle(uniq)
    dev_keys = set(uniq[:max(1, int(len(uniq) * dev_frac))])
    is_dev = np.array([g in dev_keys for g in groups])
    return teks[~is_dev], gambar[~is_dev], teks[is_dev], gambar[is_dev]

def latih_proyeksi(tr_t, tr_g, groups, val_t, dim, seed=42, max_epochs=100, batch=128, lr=1e-3, T=0.07, patience=10):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    a, b, dt, dg = split_dev(tr_t, tr_g, groups, seed=seed)
    proj = ProyeksiResidual(dim).to(device)
    opt = torch.optim.Adam(proj.parameters(), lr=lr, weight_decay=1e-4)
    f = lambda x: torch.tensor(x, dtype=torch.float32, device=device)
    A, Bg, DT, DG = f(a), f(b), f(dt), f(dg)
    best, state, bad, best_ep = float("inf"), None, 0, 0
    for ep in range(max_epochs):
        proj.train(); idx = torch.randperm(len(A), device=device)
        for i in range(0, len(A), batch):
            j = idx[i:i + batch]
            g = Bg[j] / Bg[j].norm(dim=-1, keepdim=True)
            loss = info_nce(proj(A[j]), g, T)
            opt.zero_grad(); loss.backward(); opt.step()
        proj.eval()
        with torch.no_grad():
            dl = info_nce(proj(DT), DG / DG.norm(dim=-1, keepdim=True), T).item()
        if dl < best: best, state, bad, best_ep = dl, {k: v.clone() for k, v in proj.state_dict().items()}, 0, ep + 1
        else: bad += 1
        if bad >= patience: break
    proj.load_state_dict(state); proj.eval()
    with torch.no_grad():
        return proj(f(val_t)).cpu().numpy(), best_ep
print("Fungsi siap.")

## 5. Sensitivitas semua varian: val penuh vs val bersih
Val bersih membuang 13 baris val yang berstatus pasti bocor. Selisih yang berada di dalam CI berarti kebocoran tidak menggelembungkan hasil.

In [ ]:
rows = []
for nama, v in VAL_EMB.items():
    rf = evaluasi(v["img"], v["ti"], v["te"])
    rc = evaluasi(v["img"], v["ti"], v["te"], mask=val_bersih)
    for a in ARAH:
        mf, lf, hf = ci_mean(hit1(rf[a])); mc, lc, hc = ci_mean(hit1(rc[a]))
        rows.append({"Varian": nama, "Arah": a, "R@1 penuh": mf, "CI penuh": f"[{lf:.3f}, {hf:.3f}]",
                     "R@1 bersih": mc, "CI bersih": f"[{lc:.3f}, {hc:.3f}]", "Selisih": mc - mf,
                     "Selisih di dalam CI penuh?": "Ya" if lf <= mc <= hf else "TIDAK"})
sens = pd.DataFrame(rows)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
sens.to_csv(AUDIT_DIR / "audit_sensitivitas_encoder.csv", index=False, float_format="%.4f")
display(sens.round(4))

## 6. Ablation filter pada proyeksi
Proyeksi residual dilatih ulang di bawah tiga kondisi filter dan tiga seed. Prosedur sama dengan peran 3; hanya baris train yang berbeda.

In [ ]:
HASIL = {}   # (encoder, kondisi, seed) -> (ranks_penuh, ranks_bersih, best_epoch)
t0 = time.time()
for enc in [e for e in TRAIN_EMB if e in VAL_EMB]:
    tr, v = TRAIN_EMB[enc], VAL_EMB[enc]
    for kond, keep in KONDISI.items():
        for s in SEEDS_PROJ:
            vp, ep = latih_proyeksi(tr["ti"][keep], tr["img"][keep], train_group[keep], v["ti"], tr["ti"].shape[1], seed=s)
            HASIL[(enc, kond, s)] = (evaluasi(v["img"], vp, v["te"]),
                                     evaluasi(v["img"], vp, v["te"], mask=val_bersih), ep)
    print(f"[selesai] {enc} ({time.time() - t0:.0f}s)")

In [ ]:
rows = []
for enc in sorted({k[0] for k in HASIL}):
    for kond, keep in KONDISI.items():
        rec = {"Encoder": enc, "Kondisi filter": kond, "Baris train": int(keep.sum())}
        for a_idx, nama in ((2, "teks-ID->gambar"), (0, "gambar->teks-ID")):
            for tag, pos in (("penuh", 0), ("bersih", 1)):
                x = np.array([hit1(HASIL[(enc, kond, s)][pos][ARAH[a_idx]]).mean() for s in SEEDS_PROJ])
                rec[f"R@1 {nama} {tag}"] = f"{x.mean():.4f}±{x.std():.4f}"
        rec["Best epoch"] = [HASIL[(enc, kond, s)][2] for s in SEEDS_PROJ]
        rows.append(rec)
abl = pd.DataFrame(rows)
abl.to_csv(AUDIT_DIR / "audit_ablation_filter.csv", index=False)
display(abl)

# Efek filter: selisih per query (rata-rata seed), paired bootstrap, pada val penuh
rows = []
for enc in sorted({k[0] for k in HASIL}):
    for (pos, tag) in ((0, "val penuh"), (1, "val bersih")):
        for a_idx, nama in ((2, "teks-ID->gambar"), (0, "gambar->teks-ID")):
            hv = lambda kond: np.mean([hit1(HASIL[(enc, kond, s)][pos][ARAH[a_idx]]) for s in SEEDS_PROJ], axis=0)
            for kond in ("filter caption (peran 3)", "filter audit (peran 1)"):
                m, lo, hi = ci_mean(hv(kond) - hv("tanpa filter"))
                rows.append({"Encoder": enc, "Evaluasi": tag, "Arah": nama, "Perbandingan": f"{kond} vs tanpa filter",
                             "Δ R@1": m, "CI 2.5%": lo, "CI 97.5%": hi,
                             "CI mencakup 0?": "Ya (tidak ada efek terbukti)" if lo <= 0 <= hi else "TIDAK"})
efek = pd.DataFrame(rows)
efek.to_csv(AUDIT_DIR / "audit_efek_filter.csv", index=False, float_format="%.4f")
display(efek.round(4))

In [ ]:
# Grafik: R@1 teks-ID->gambar per kondisi filter (rata-rata seed), val penuh vs bersih
encs = sorted({k[0] for k in HASIL})
fig, axes = plt.subplots(1, max(1, len(encs)), figsize=(5.5 * max(1, len(encs)), 4), squeeze=False)
for ax, enc in zip(axes[0], encs):
    w = 0.38
    for j, (pos, tag) in enumerate(((0, "val penuh"), (1, "val bersih"))):
        mu = [np.mean([hit1(HASIL[(enc, k, s)][pos][ARAH[2]]).mean() for s in SEEDS_PROJ]) for k in KONDISI]
        sd = [np.std([hit1(HASIL[(enc, k, s)][pos][ARAH[2]]).mean() for s in SEEDS_PROJ]) for k in KONDISI]
        ax.bar(np.arange(len(KONDISI)) + (j - 0.5) * w, mu, w, yerr=sd, capsize=3, label=tag)
    ax.set_xticks(range(len(KONDISI))); ax.set_xticklabels(["tanpa\nfilter", "filter\ncaption", "filter\naudit"])
    ax.set_title(f"{enc} + proyeksi"); ax.set_ylabel("R@1 teks-ID → gambar"); ax.legend()
plt.tight_layout(); plt.show()

## 7. Kesimpulan dan limitasi

*(Diisi setelah semua sel di atas dijalankan di Colab.)*

**Aturan membaca hasil**
- Selisih penuh vs bersih (bagian 5) di dalam CI: kebocoran val tidak menggelembungkan angka.
- Efek filter (bagian 6) dengan CI mencakup 0: filter train tidak mengubah hasil secara terbukti.
- Peningkatan varian yang lebih kecil dari CI tidak boleh dinyatakan lebih baik.

**Limitasi yang tidak dapat dihilangkan**
1. **Kontaminasi pra-latih.** Encoder pra-latih (CLIP, SigLIP 2 / WebLI, NLLB-CLIP) mungkin pernah melihat gambar atau teks yang mirip dataset ini. Tidak dapat diperiksa dari dalam proyek; kenaikan besar SigLIP 2 (R@1 teks-ID → gambar 0,196 → 0,498) perlu dibaca dengan kehati-hatian ini.
2. **Pemilihan encoder terbaik memakai val**, hanya dengan dua kandidat dan selisih jauh di atas CI.
3. Val hanya 275 baris: CI R@1 sekitar ±5 poin.